In [0]:
import pandas as pd
import random
import json
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql.types import *
from pyspark.sql.functions import count,col


In [0]:
dbutils.widgets.text("num_orders","25","Num of Orders to Generate")
Num_Orders = int(dbutils.widgets.get("num_orders"))
print(f"verifying the datatype of the Num_Orders - {type(Num_Orders)}")
print(f"Num_Orders = {Num_Orders}")


CATALOG = "restaurant_catalog"
VOL_PATH = f"/Volumes/{CATALOG}/landing/raw_files"
TARGET_TABLE = f"{CATALOG}.landing.streaming_orders_raw"

In [0]:
def read_csv_from_volume(folder_name:str) -> pd.DataFrame:
    """
    Reads a single-part CSV from a Unity Catalog Volume subfolder.
    Spark writes CSVs as part files (part-00000-*.csv) — this function
    finds the actual .csv file automatically using dbutils.fs.ls().
    """
    folder_path = f"{VOL_PATH}/{folder_name}"
    all_files = dbutils.fs.ls(folder_path)

    csv_files = [f.path for f in all_files if f.name.endswith(".csv")]
    print(f" List of csv files in {folder_path} are {csv_files}")
    if not csv_files:
        raise FileNotFoundError(f"No .csv file in {folder_path}")

    local_path = csv_files[0].replace("dbfs:","")
    return pd.read_csv(local_path)


    

In [0]:
# Load all 3 reference datasets

restaurants_pd = read_csv_from_volume("restaurants")
customers_pd = read_csv_from_volume("customers")
menu_pd = read_csv_from_volume("menu_items")

In [0]:
# display(restaurants_pd)

# display(menu_pd.head(5))

menu_pd_DF = spark.createDataFrame(menu_pd)
# display(menu_pd_DF.groupby("restaurant_id").agg(count("item_id")))

display(menu_pd_DF.filter( (col("restaurant_id")=="R001") & (col("is_available")=="True")))

In [0]:
# Build a dict: restaurant_id → [list of available menu item dicts]
# This lets us quickly look up "what can be ordered at R001?" during order generation
rest_menu = {}
for _, row in menu_pd[menu_pd["is_available"] == True].iterrows():
    rest_menu.setdefault(row["restaurant_id"], []).append(row.to_dict())

# Plain Python lists for random.choice() — fastest for small data
restaurants_list = restaurants_pd["restaurant_id"].tolist()
customers_list   = customers_pd["customer_id"].tolist()

print(f"✅ Reference data loaded:")
print(f"   Restaurants : {len(restaurants_list)}")
print(f"   Customers   : {len(customers_list)}")
print(f"   Menu items  : {len(menu_pd)} total | {sum(len(v) for v in rest_menu.values())} available")
print(f"   Per restaurant: { {k: len(v) for k,v in rest_menu.items()} }")

In [0]:
# ─── CELL 3: CREATE STREAMING SOURCE TABLE (ONCE, IDEMPOTENT) ─────────────────
#
# WHAT YOU'RE DOING:
#   Creating the Delta table that acts as your "Event Hub topic".
#   The DLT Bronze pipeline will read this table as a continuous stream
#   using spark.readStream.format("delta").
#
# WHY CREATE TABLE IF NOT EXISTS:
#   "IF NOT EXISTS" makes this idempotent — safe to run multiple times.
#   First run: creates the table. Subsequent runs: silently skips.
#   This is the production standard. Never use CREATE TABLE without
#   IF NOT EXISTS in setup notebooks — it would fail on the second run.
#
# ABOUT delta.enableChangeDataFeed = true:
#   Change Data Feed (CDF) tracks row-level inserts/updates/deletes
#   with metadata (operation type + commit timestamp).
#   Enabling it here is a production best practice — it doesn't cost
#   much extra storage but gives you full CDC capability if a downstream
#   consumer ever needs to know "what changed since last read?".
#   The DLT pipeline does NOT require CDF — but it's good to have.
#
# WHY COLUMN COMMENTS:
#   In Unity Catalog, column comments appear in the Data Explorer and
#   Catalog UI. Any analyst or downstream engineer can hover over a column
#   and know what it means. This is a data governance production standard.
# ──────────────────────────────────────────────────────────────────────────────

spark.sql(
    f"""
    create table if not exists {TARGET_TABLE} (
        order_id        STRING  COMMENT 'Unique live order ID format: ORD_LIVE_<YYYYMMDDHHMMSS>_<seq>',
        timestamp       STRING  COMMENT 'Order creation time as ISO 8601 string: YYYY-MM-DD HH:MM:SS',
        restaurant_id   STRING  COMMENT 'FK to dim_restaurants — must be one of R001..R005',
        customer_id     STRING  COMMENT 'FK to dim_customers — format: C00001..C00500',
        items           STRING  COMMENT 'JSON array of ordered items. Each item: item_id, item_name, category, quantity, unit_price, subtotal',
        total_amount    DOUBLE  COMMENT 'Total order value in AED (sum of all item subtotals)',
        payment_method  STRING  COMMENT 'Payment type: card | cash | wallet',
        order_type      STRING  COMMENT 'How customer received order: dine_in | takeaway | delivery',
        order_status    STRING  COMMENT 'Current order state: completed | pending | cancelled'    )
        using delta
        comment 'Sumulated POS streaming Source, replacing Azure event hub'
        tblproperties ('delta.enableChangeDataFeed'='true')
    """
)

print(f"✅ Streaming source table ready: {TARGET_TABLE}")
print(f"   The DLT Bronze pipeline will read this as a Delta stream.")

In [0]:
from delta.tables import DeltaTable

# df = DeltaTable.forName(spark,TARGET_TABLE)
df = spark.sql(f"select * from {TARGET_TABLE}")
display(df)

In [0]:
# ─── CELL 4: ORDER GENERATION FUNCTION ────────────────────────────────────────
#
# WHAT YOU'RE DOING:
#   Defining a function that generates ONE realistic order as a Spark Row.
#   A Spark Row is like a single-row namedtuple — it maps to one row
#   in a DataFrame and eventually one row in the Delta table.
#
# WHY random.choices() WITH WEIGHTS FOR order_status:
#   In a real POS system, most orders complete successfully. Cancellations
#   are rare (5%) and pending states are transient (15%).
#   Using weights=[80, 15, 5] instead of equal probability makes the
#   data distribution realistic — which matters for the Gold aggregations
#   and dashboard metrics downstream.
#
# WHY order_id INCLUDES THE TIMESTAMP:
#   ORD_LIVE_20260411221500_0001 is a self-describing ID.
#   Anyone looking at the order_id immediately knows:
#   - It's a LIVE order (vs ORD0000001 from historical batch)
#   - When it was generated (2026-04-11 22:15:00)
#   This also ensures global uniqueness across multiple producer runs.
# ──────────────────────────────────────────────────────────────────────────────

schema = StructType([
    StructField("order_id",       StringType()),
    StructField("timestamp",      StringType()),
    StructField("restaurant_id",  StringType()),
    StructField("customer_id",    StringType()),
    StructField("items",          StringType()),
    StructField("total_amount",   DoubleType()),
    StructField("payment_method", StringType()),
    StructField("order_type",     StringType()),
    StructField("order_status",   StringType()),
])

payment_methods = ["card", "cash", "wallet"]
order_types     = ["dine_in", "takeaway", "delivery"]

def generate_order(seq: int) -> Row:
    """
    Generates one realistic live order.
    seq: sequence number within this producer run (0 to NUM_ORDERS-1).
    Used to ensure unique order_ids even if two orders land at the same second.
    """
    restaurant_id = random.choice(restaurants_list)
    customer_id   = random.choice(customers_list)
    ts            = datetime.now()
    items_raw     = rest_menu.get(restaurant_id, [])

    if items_raw:
        # Pick 1 to 4 random available items from this restaurant's menu
        num_items = random.randint(1, min(4, len(items_raw)))
        chosen    = random.sample(items_raw, num_items)

        # Build the items JSON array — one dict per ordered item
        items_out = []
        for it in chosen:
            qty = random.randint(1, 3)
            items_out.append({
                "item_id"   : it["item_id"],
                "item_name" : it["item_name"],
                "category"  : it["category"],
                "quantity"  : qty,
                "unit_price": float(it["unit_price"]),
                "subtotal"  : round(float(it["unit_price"]) * qty, 2)
            })
        total = round(sum(i["subtotal"] for i in items_out), 2)
    else:
        # Fallback: if rest_menu somehow has no entry for this restaurant
        items_out = [{"item_id":"ITEM0001","item_name":"Butter Chicken",
                      "category":"Main Course","quantity":2,"unit_price":55.0,"subtotal":110.0}]
        total     = 110.0

    return Row(
        order_id       = f"ORD_LIVE_{ts.strftime('%Y%m%d%H%M%S')}_{seq:04d}",
        timestamp      = ts.strftime("%Y-%m-%d %H:%M:%S"),
        restaurant_id  = restaurant_id,
        customer_id    = customer_id,
        items          = json.dumps(items_out),
        total_amount   = total,
        payment_method = random.choice(payment_methods),
        order_type     = random.choice(order_types),
        order_status   = random.choices(
            ["completed", "pending", "cancelled"],
            weights=[80, 15, 5]       # 80% complete, 15% pending, 5% cancelled
        )[0],
    )

print("✅ Order generation function defined and ready.")

In [0]:
# ─── CELL 5: GENERATE + APPEND ORDERS TO DELTA TABLE ─────────────────────────
#
# WHAT YOU'RE DOING:
#   1. Generating NUM_ORDERS order Rows using the function from Cell 4
#   2. Converting them to a Spark DataFrame using the explicit schema
#   3. Writing to the Delta table using mode("append")
#
# WHY spark.createDataFrame() WITH EXPLICIT SCHEMA:
#   If you let Spark infer the schema from the data, it might guess wrong.
#   For example, total_amount could be inferred as FLOAT instead of DOUBLE.
#   Explicit schema = guaranteed, consistent column types every single run.
#   This is especially important for the DLT pipeline reading this table —
#   schema mismatches between runs cause pipeline failures.
#
# WHY mode("append") AND NOT mode("overwrite"):
#   This table is a streaming source. Every write is a new Delta commit.
#   The DLT pipeline tracks which commits it has already processed using
#   its checkpoint. If you used overwrite, you'd delete old orders that
#   the pipeline may not have processed yet — breaking the stream.
#   append adds a new Delta version, which the streaming reader picks up
#   on its next trigger. This is exactly how Kafka producers work:
#   they append to a topic, never overwrite.
# ──────────────────────────────────────────────────────────────────────────────

# Generate all orders as Python Row objects (driver-side, fast)
new_orders = [generate_order(i) for i in range(Num_Orders)]

# Convert to Spark DataFrame — now distributable across workers if needed
df_new = spark.createDataFrame(new_orders, schema)

# Append to Delta table — creates a new Delta commit (new "version")
# The DLT Bronze pipeline's readStream will detect this new version
# on its next trigger and process exactly these new rows
df_new.write.mode("append").saveAsTable(TARGET_TABLE)

# Verify by counting total rows in the table
total_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {TARGET_TABLE}").first().cnt
print(f"✅ Successfully pushed {Num_Orders} new orders to {TARGET_TABLE}")
print(f"📊 Total orders in streaming source table: {total_count:,}")
print(f"   Run this notebook again to push more batches.")
print(f"   Each run creates a new Delta commit → DLT Bronze will process it.")

In [0]:
# df = DeltaTable.forName(spark,TARGET_TABLE) 
df = spark.table(TARGET_TABLE)
display(df)